# Offline SageMaker local mode — CatBoost training

Fully offline (no AWS) training against a moto-backed S3/STS endpoint using the `sagemaker-local` library. The training job runs in a local Docker container.

Dataset: `diabetes` (regression), loaded inside the container from scikit-learn. The built `sagemaker-local:latest` image already includes catboost 1.2.7.

In [1]:
import os

from sagemaker.deserializers import JSONDeserializer
from sagemaker.estimator import Estimator
from sagemaker.serializers import JSONSerializer
from sagemaker_local.config import config_from_env
from sagemaker_local.session import make_local_session

cfg = config_from_env()
boto_session, sm_session = make_local_session(cfg)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/jupyter/.config/sagemaker/config.yaml


## Train

`est.fit()` runs synchronously in a local container; the script `train.py` loads `diabetes` and writes `model.joblib` to `/opt/ml/model`.

The generic `Estimator` (bring-your-own-container) is pointed at the local image via `image_uri` so no AWS image is pulled.

In [2]:
est = Estimator(
    entry_point="train.py",
    source_dir=os.path.join(
        os.environ.get("SAGEMAKER_LOCAL_REPO_PATH", "/workspace"),
        "projects",
        "sagemaker_catboost",
        "src",
        "sagemaker_catboost",
    ),
    image_uri=cfg.image_tag,
    role=cfg.role_arn,
    instance_type="local",
    instance_count=1,
    sagemaker_session=sm_session,
    output_path=f"s3://{cfg.bucket}/models",
    hyperparameters={"dataset": "diabetes"},
)
est.fit()

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.TelemetryOptOut


INFO:sagemaker:Creating training-job with name: sagemaker-local-2026-09-01-01-03-20-049


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.TelemetryOptOut


INFO:sagemaker.local.local_session:Starting training job


...

time="2026-09-01T01:03:20Z" level=warning msg="/tmp/tmp3kppnwok/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-09-01T01:03:20Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmp3kppnwok\".\nSet `external: true` to use an existing network"
 Container lia68ioaul-algo-1-hqglt  Creating
 Container lia68ioaul-algo-1-hqglt  Created
Attaching to lia68ioaul-algo-1-hqglt


lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,547 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code is not empty, abandoning unzipping sourcedir.tar.gz
lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,547 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,548 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,554 sagemaker-training-toolkit INFO     instance_groups entry not present in resource_config
lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,558 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,558 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:21,564 sagemaker-training-toolkit INFO     instance_groups entry not present in resource_config
lia68

lia68ioaul-algo-1-hqglt  | 2026-09-01 01:03:22,871 sagemaker-training-toolkit INFO     Reporting training SUCCESS
lia68ioaul-algo-1-hqglt exited with code 0
Aborting on container exit...


INFO:sagemaker.local.image:===== Job Complete =====


 Container lia68ioaul-algo-1-hqglt  Stopping
 Container lia68ioaul-algo-1-hqglt  Stopped


## Deploy & predict

Inputs are 2-D JSON arrays so the default serving input handler parses them correctly.

In [3]:
predictor = est.deploy(
    initial_instance_count=1,
    instance_type="local",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)
sample = [0.05, 0.08, 0.0, 0.02, 0.04, 0.06, 0.0, 0.0, 0.03, 0.05]
result = predictor.predict([sample])
print("predicted disease progression:", result)

INFO:sagemaker:Creating model with name: sagemaker-local-2026-09-01-01-03-23-135


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.TelemetryOptOut


INFO:sagemaker:Creating endpoint-config with name sagemaker-local-2026-09-01-01-03-23-135


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


...

0dzlynclw0-algo-1-jk75h  | [2026-09-01 01:03:25 +0000] [7] [INFO] Starting gunicorn 23.0.0
0dzlynclw0-algo-1-jk75h  | [2026-09-01 01:03:25 +0000] [7] [INFO] Listening at: http://0.0.0.0:8080 (7)
0dzlynclw0-algo-1-jk75h  | [2026-09-01 01:03:25 +0000] [7] [INFO] Using worker: sync
0dzlynclw0-algo-1-jk75h  | [2026-09-01 01:03:25 +0000] [15] [INFO] Booting worker with pid: 15


INFO:sagemaker.local.entities:Checking if serving container is up, attempt: 10


!

INFO:sagemaker_local.patches:resolved docker host gateway: 172.22.0.1


predicted disease progression: [162.5642646723095]


## Cleanup

Delete the endpoint so the serving container is released and port `8080` is freed.

In [4]:
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: sagemaker-local-2026-09-01-01-03-23-135


INFO:sagemaker:Deleting endpoint with name: sagemaker-local-2026-09-01-01-03-23-135
